# Data Profiling — Freemarket Relationship-Network

**Purpose.** Understand the four raw sources in `data/` *before* building the Bronze → Silver → Gold
pipeline. This notebook is **read-only reconnaissance**: it does not write to `warehouse.duckdb` and it
does not model anything. It answers "what have we actually got, and where will it hurt?"

Sources profiled (see `SETUP.md`, `BRIEF.md`, `docs/build_protocol.md`):

| File | What it is | Key |
|---|---|---|
| `transactional_data_jul25_dec25.xlsx` | Core facts — 4 sheets: `Deposit`, `Withdrawals`, `Counterparty`, `Fees` | `Transaction ID`, `CP ID`, `FeeId` |
| `companies.json` | Direct companies (the entities that transact) | `dcId` |
| `groups.json` | Parent groups (the hierarchy companies roll up into) | `groupId` |
| `exchange_rates.json` | Time-versioned FX rates to GBP (~19 MB) | currency code |

**What we look for:** shape & schema, key uniqueness, nulls, value distributions, date/currency coverage,
cross-source referential integrity, and the FX as-of feasibility (currencies used vs currencies priced,
transaction window vs FX coverage). Each section ends with 🔎 **profiling notes** — the facts a modelling
decision will hang off.

Run top-to-bottom. Uses DuckDB (the project engine) for reads that mirror the pipeline, and pandas for display.

## 0. Setup

In [1]:
import json, os, warnings
from pathlib import Path

import duckdb
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 60)

# Resolve the data/ folder whether the notebook runs from notebooks/, submission/ or repo root.
HERE = Path.cwd()
DATA = next((p / "data" for p in [HERE, *HERE.parents] if (p / "data").is_dir()), None)
assert DATA is not None, "Could not locate the data/ directory."
XLSX = DATA / "transactional_data_jul25_dec25.xlsx"
print("data dir:", DATA)
for f in sorted(DATA.glob("*")):
    print(f"  {f.name:45s} {f.stat().st_size/1e6:8.2f} MB")

# In-memory DuckDB — profiling only, we never touch warehouse.duckdb here.
con = duckdb.connect(":memory:")
con.execute("INSTALL excel; LOAD excel;")
FX_READ = f"read_json_auto('{XLSX.parent / 'exchange_rates.json'}', maximum_object_size=134217728)"
print("\nduckdb", duckdb.__version__)

data dir: /Users/hallabehery/Documents/My Projects/FM-Data-Engineer-Test-HS/data
  companies.json                                    0.06 MB
  exchange_rates.json                              18.05 MB
  groups.json                                       0.02 MB
  transactional_data_jul25_dec25.xlsx               1.76 MB

duckdb 1.5.4


In [2]:
def profile_frame(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Per-column profile: dtype, non-null, null %, distinct, and a small sample of values."""
    n = len(df)
    rows = []
    for c in df.columns:
        s = df[c]
        nn = s.notna().sum()
        distinct = s.nunique(dropna=True)
        sample = ", ".join(map(str, pd.Series(s.dropna().unique()[:3])))
        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "non_null": nn,
            "null_%": round(100 * (n - nn) / n, 1) if n else 0.0,
            "distinct": distinct,
            "sample_values": (sample[:70] + "…") if len(sample) > 70 else sample,
        })
    out = pd.DataFrame(rows)
    print(f"{name}: {n:,} rows × {df.shape[1]} cols")
    return out

## 1. Transactional workbook — `transactional_data_jul25_dec25.xlsx`

Four sheets. `Deposit` / `Withdrawals` are the fact rows; `Counterparty` is a dimension keyed on `CP ID`;
`Fees` links back to transactions via `Link Id`. Note the build protocol's warning: **Deposit and
Withdrawals do not share column order — align on name, never position.**

In [3]:
sheets = {}
for sheet in ("Deposit", "Withdrawals", "Counterparty", "Fees"):
    sheets[sheet] = pd.read_excel(XLSX, sheet_name=sheet)
{k: v.shape for k, v in sheets.items()}

{'Deposit': (6383, 12),
 'Withdrawals': (6599, 12),
 'Counterparty': (1585, 7),
 'Fees': (21921, 7)}

### 1a. `Deposit`

In [4]:
profile_frame(sheets["Deposit"], "Deposit")

Deposit: 6,383 rows × 12 cols


,column,dtype,non_null,null_%,distinct,sample_values
0,Freemarket Entity,str,6383,0.0,2,"UK, EEA"
1,Transaction Type,str,6383,0.0,1,Deposit
2,Deal ID/DC ID,int64,6383,0.0,27,"234939159769, 263002682585, 234850008297"
3,Account ID,int64,6383,0.0,48,"16472, 16968, 16627"
4,Transaction ID,int64,6383,0.0,6383,"371637, 374086, 367878"
5,Tx Date,datetime64[us],6383,0.0,139,"2025-10-24 00:00:00, 2025-10-31 00:00:00, 2025-10-14 00:..."
6,Tx Time,object,6383,0.0,6379,"11:04:52.740000, 14:57:23.760000, 13:43:45.990000"
7,Tx Currency,str,6383,0.0,9,"GBP, USD, EUR"
8,Tx Value (CCY),float64,6383,0.0,3319,"5424.988391519951, 19752.81041789988, 32328485.28861931"
9,Counterparty,str,6383,0.0,361,"CP1, CP4, CP5"


In [5]:
sheets["Deposit"].head(5)

,Freemarket Entity,Transaction Type,Deal ID/DC ID,Account ID,Transaction ID,Tx Date,Tx Time,Tx Currency,Tx Value (CCY),Counterparty,Tx Month,Scheme
0,UK,Deposit,234939159769,16472,371637,2025-10-24,11:04:52.740000,GBP,5.424988e+03,CP1,2025-10-01,FasterPayments
1,UK,Deposit,234939159769,16472,374086,2025-10-31,14:57:23.760000,GBP,1.975281e+04,CP4,2025-10-01,FasterPayments
2,UK,Deposit,234939159769,16472,367878,2025-10-14,13:43:45.990000,USD,3.232849e+07,CP5,2025-10-01,Swift
3,UK,Deposit,234939159769,16472,343316,2025-07-30,13:38:31.593000,USD,2.438954e+06,CP6,2025-07-01,Swift
4,UK,Deposit,234939159769,16472,341662,2025-07-24,10:39:47.117000,USD,2.736424e+05,CP5,2025-07-01,Swift


### 1b. `Withdrawals`

In [6]:
profile_frame(sheets["Withdrawals"], "Withdrawals")

Withdrawals: 6,599 rows × 12 cols


,column,dtype,non_null,null_%,distinct,sample_values
0,Freemarket Entity,str,6599,0.0,2,"UK, EEA"
1,Transaction Type,str,6599,0.0,1,Withdrawal
2,Deal ID/DC ID,int64,6599,0.0,34,"262995992770, 262979201222, 262994262242"
3,Account ID,int64,6599,0.0,56,"14445, 17329, 14459"
4,Transaction ID,int64,6599,0.0,6599,"380113, 427636, 372371"
5,Tx Date,datetime64[us],6599,0.0,133,"2025-08-26 00:00:00, 2025-11-13 00:00:00, 2025-07-24 00:..."
6,Tx Time,object,6599,0.0,6596,"08:54:24.523000, 15:01:36.070000, 14:55:01.443000"
7,Tx Currency,str,6599,0.0,9,"EUR, USD, GBP"
8,Tx Value (CCY),float64,6599,0.0,6599,"56.87839388217648, 11628.63124023862, 677.651644282242"
9,Tx Month,datetime64[us],6599,0.0,6,"2025-08-01 00:00:00, 2025-11-01 00:00:00, 2025-07-01 00:..."


In [7]:
sheets["Withdrawals"].head(5)

,Freemarket Entity,Transaction Type,Deal ID/DC ID,Account ID,Transaction ID,Tx Date,Tx Time,Tx Currency,Tx Value (CCY),Tx Month,Counterparty,Scheme
0,UK,Withdrawal,262995992770,14445,380113,2025-08-26,08:54:24.523000,EUR,56.878394,2025-08-01,CP650,Sepa
1,UK,Withdrawal,262979201222,17329,427636,2025-11-13,15:01:36.070000,EUR,11628.631240,2025-11-01,CP904,Sepa
2,UK,Withdrawal,262994262242,14459,372371,2025-07-24,14:55:01.443000,EUR,677.651644,2025-07-01,CP906,Sepa
3,UK,Withdrawal,262994262242,14459,412208,2025-10-28,08:41:10.930000,EUR,243.287539,2025-10-01,CP907,Sepa
4,UK,Withdrawal,262994262242,14459,372339,2025-07-24,14:51:36.927000,EUR,18.783131,2025-07-01,CP912,Sepa


### 1c. Deposit vs Withdrawals — column-order check
The protocol says these sheets differ in column order. Confirm it, and confirm the *sets* match so a name-aligned union is safe.

In [8]:
dep_cols, wd_cols = list(sheets["Deposit"].columns), list(sheets["Withdrawals"].columns)
print("Deposit cols    :", dep_cols)
print("Withdrawals cols:", wd_cols)
print("\nSame order?      ", dep_cols == wd_cols)
print("Same set?        ", set(dep_cols) == set(wd_cols))
print("Only in Deposit  :", set(dep_cols) - set(wd_cols))
print("Only in Withdraw :", set(wd_cols) - set(dep_cols))
# position-by-position diff
pd.DataFrame({"pos": range(len(dep_cols)), "Deposit": dep_cols, "Withdrawals": wd_cols,
              "same_position": [d == w for d, w in zip(dep_cols, wd_cols)]})

Deposit cols    : ['Freemarket Entity', 'Transaction Type', 'Deal ID/DC ID', 'Account ID', 'Transaction ID', 'Tx Date', 'Tx Time', 'Tx Currency', 'Tx Value (CCY)', 'Counterparty', 'Tx Month', 'Scheme']
Withdrawals cols: ['Freemarket Entity', 'Transaction Type', 'Deal ID/DC ID', 'Account ID', 'Transaction ID', 'Tx Date', 'Tx Time', 'Tx Currency', 'Tx Value (CCY)', 'Tx Month', 'Counterparty', 'Scheme']

Same order?       False
Same set?         True
Only in Deposit  : set()
Only in Withdraw : set()


,pos,Deposit,Withdrawals,same_position
0,0,Freemarket Entity,Freemarket Entity,True
1,1,Transaction Type,Transaction Type,True
2,2,Deal ID/DC ID,Deal ID/DC ID,True
3,3,Account ID,Account ID,True
4,4,Transaction ID,Transaction ID,True
5,5,Tx Date,Tx Date,True
6,6,Tx Time,Tx Time,True
7,7,Tx Currency,Tx Currency,True
8,8,Tx Value (CCY),Tx Value (CCY),True
9,9,Counterparty,Tx Month,False


### 1d. Fact-row distributions — currency, month, entity, scheme, type
What currencies must FX handle? Does the data really span Jul–Dec 2025? Are there odd entities / schemes?

In [9]:
for sheet in ("Deposit", "Withdrawals"):
    df = sheets[sheet]
    print(f"\n========== {sheet} ==========")
    print("Tx Date range :", df["Tx Date"].min(), "→", df["Tx Date"].max())
    print("Tx Month vals :", sorted(df["Tx Month"].dropna().astype(str).unique()))
    print("Currencies    :", df["Tx Currency"].value_counts().to_dict())
    print("Entities      :", df["Freemarket Entity"].value_counts().to_dict())
    print("Schemes       :", df["Scheme"].value_counts().to_dict())
    print("Txn Types     :", df["Transaction Type"].value_counts().to_dict())


========== Deposit ==========
Tx Date range : 2025-07-01 00:00:00 → 2025-12-31 00:00:00
Tx Month vals : ['2025-07-01', '2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01']
Currencies    : {'EUR': 3405, 'USD': 1829, 'GBP': 535, 'CAD': 204, 'SEK': 185, 'NOK': 177, 'DKK': 39, 'CHF': 5, 'AED': 4}
Entities      : {'UK': 6272, 'EEA': 111}
Schemes       : {'Swift': 2531, 'SepaInstant': 2468, 'Sepa': 758, 'FasterPayments': 515, 'Target2': 87, 'Chaps': 18}
Txn Types     : {'Deposit': 6383}

========== Withdrawals ==========
Tx Date range : 2025-07-01 00:00:00 → 2025-12-31 00:00:00
Tx Month vals : ['2025-07-01', '2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01']
Currencies    : {'USD': 2952, 'EUR': 2816, 'GBP': 487, 'SEK': 156, 'CAD': 142, 'CHF': 30, 'AED': 12, 'DKK': 2, 'AUD': 2}
Entities      : {'UK': 6555, 'EEA': 44}
Schemes       : {'Swift': 3598, 'Sepa': 2437, 'FasterPayments': 471, 'Chaps': 3}
Txn Types     : {'Withdrawal': 6599}


### 1e. Tx value sanity — negatives, zeros, magnitude by currency
Amounts are in native currency (`Tx Value (CCY)`), so magnitudes differ wildly across currencies. Flag non-positive values.

In [10]:
for sheet in ("Deposit", "Withdrawals"):
    df = sheets[sheet]
    v = df["Tx Value (CCY)"]
    print(f"{sheet:12s} min={v.min():,.2f}  max={v.max():,.2e}  zeros={(v==0).sum()}  negatives={(v<0).sum()}  nulls={v.isna().sum()}")
print("\nMedian native value by currency (Deposit):")
sheets["Deposit"].groupby("Tx Currency")["Tx Value (CCY)"].agg(["count", "median", "max"]).sort_values("count", ascending=False)

Deposit      min=0.21  max=7.77e+08  zeros=0  negatives=0  nulls=0
Withdrawals  min=0.01  max=2.01e+10  zeros=0  negatives=0  nulls=0

Median native value by currency (Deposit):


,count,median,max
Tx Currency,,,
EUR,3405,99994.000,4.664089e+08
USD,1829,500000.000,7.774086e+08
GBP,535,50000.000,2.027838e+07
CAD,204,397371.875,1.110120e+08
SEK,185,1282875.600,1.860364e+08
NOK,177,1724766.190,6.979615e+07
DKK,39,300578.710,2.068654e+06
CHF,5,50000.000,6.546431e+05
AED,4,750.000,4.090822e+05


### 1f. `Transaction ID` uniqueness & Deposit/Withdrawal overlap
`Transaction ID` is the natural key. Check duplicates within each sheet and whether any ID appears in *both* (which would break a naive union).

In [11]:
dep_ids, wd_ids = sheets["Deposit"]["Transaction ID"], sheets["Withdrawals"]["Transaction ID"]
print("Deposit    : rows", len(dep_ids), " distinct", dep_ids.nunique(), " dupes", dep_ids.duplicated().sum())
print("Withdrawals: rows", len(wd_ids), " distinct", wd_ids.nunique(), " dupes", wd_ids.duplicated().sum())
overlap = set(dep_ids) & set(wd_ids)
print("IDs in BOTH sheets:", len(overlap), (sorted(overlap)[:10] if overlap else ""))

Deposit    : rows 6383  distinct 6383  dupes 0
Withdrawals: rows 6599  distinct 6599  dupes 0
IDs in BOTH sheets: 306 [366182, 366183, 366661, 366680, 366693, 366723, 366754, 366756, 367330, 367410]


### 1g. `Counterparty` dimension

In [12]:
profile_frame(sheets["Counterparty"], "Counterparty")

Counterparty: 1,585 rows × 7 cols


,column,dtype,non_null,null_%,distinct,sample_values
0,CP ID,str,1585,0.0,1585,"CP1, CP2, CP3"
1,CP name,str,1585,0.0,412,"Sable Solutions, Koba2Maya Treasury, Aurora Logistics"
2,CP vertical,str,1097,30.8,18,"E-commerce, Online Gaming & Gambling, Logistics"
3,CP website,str,872,45.0,364,"https://www.beaconpartners.com, https://www.verdantpartn..."
4,CP business desc,str,541,65.9,493,Sable Solutions is a counterparty active in the e-commer...
5,Group ID,float64,222,86.0,13,"192230405343.0, 192402023626.0, 192212183238.0"
6,DC Id,float64,222,86.0,13,"262956506344.0, 262790438097.0, 262979201222.0"


In [13]:
cp = sheets["Counterparty"]
print("CP ID distinct  :", cp["CP ID"].nunique(), "of", len(cp), "rows; dupes:", cp["CP ID"].duplicated().sum())
print("Has Group ID    :", cp["Group ID"].notna().sum(), f"({100*cp['Group ID'].notna().mean():.0f}%)")
print("Has DC Id       :", cp["DC Id"].notna().sum(), f"({100*cp['DC Id'].notna().mean():.0f}%)")
print("Has neither     :", ((cp['Group ID'].isna()) & (cp['DC Id'].isna())).sum(), "→ these are the standalone 'diamond' nodes")
print("Verticals       :", cp["CP vertical"].nunique(), "distinct")
cp.head(5)

CP ID distinct  : 1585 of 1585 rows; dupes: 0
Has Group ID    : 222 (14%)
Has DC Id       : 222 (14%)
Has neither     : 1363 → these are the standalone 'diamond' nodes
Verticals       : 18 distinct


,CP ID,CP name,CP vertical,CP website,CP business desc,Group ID,DC Id
0,CP1,Sable Solutions,E-commerce,NaN,Sable Solutions is a counterparty active in the e-commer...,NaN,NaN
1,CP2,Koba2Maya Treasury,Online Gaming & Gambling,NaN,NaN,1.922304e+11,2.629565e+11
2,CP3,Aurora Logistics,Logistics,NaN,NaN,NaN,NaN
3,CP4,Cascade Solutions,Crypto Exchange,NaN,NaN,NaN,NaN
4,CP5,Beacon Partners,NaN,https://www.beaconpartners.com,Beacon Partners is a counterparty active in the foreign ...,NaN,NaN


### 1h. `Fees`

In [14]:
profile_frame(sheets["Fees"], "Fees")

Fees: 21,921 rows × 7 cols


,column,dtype,non_null,null_%,distinct,sample_values
0,FeeId,str,21921,0.0,21921,"FEE00000001, FEE00000002, FEE00000003"
1,Type,str,21921,0.0,2,"Deposit, Withdrawal"
2,Date,datetime64[us],21921,0.0,141,"2025-10-24 00:00:00, 2025-10-31 00:00:00, 2025-10-14 00:..."
3,FeeDetail,str,21921,0.0,2,"Flat Fee, % Fee"
4,Fee amount (CCY),float64,21921,0.0,6122,"4.89, 16.11, 17.79"
5,Fee currency,str,21879,0.2,9,"GBP, USD, EUR"
6,Link Id,int64,21921,0.0,12676,"371637, 374086, 367878"


In [15]:
fees = sheets["Fees"]
print("Date range   :", fees["Date"].min(), "→", fees["Date"].max())
print("Fee currency :", fees["Fee currency"].value_counts().to_dict())
print("Fee Type     :", fees["Type"].value_counts().to_dict())
print("FeeDetail    :", fees["FeeDetail"].value_counts().to_dict())
print("Fee amount   : min", fees["Fee amount (CCY)"].min(), " max", fees["Fee amount (CCY)"].max(),
      " zeros", (fees["Fee amount (CCY)"]==0).sum(), " neg", (fees["Fee amount (CCY)"]<0).sum())
print("FeeId dupes  :", fees["FeeId"].duplicated().sum())
fees.head(5)

Date range   : 2025-07-01 00:00:00 → 2025-12-31 00:00:00
Fee currency : {'EUR': 10485, 'USD': 8075, 'GBP': 1720, 'CAD': 595, 'SEK': 570, 'NOK': 296, 'DKK': 71, 'CHF': 50, 'AED': 17}
Fee Type     : {'Withdrawal': 11099, 'Deposit': 10822}
FeeDetail    : {'% Fee': 12982, 'Flat Fee': 8939}
Fee amount   : min 0.0  max 191279.12  zeros 496  neg 0
FeeId dupes  : 0


,FeeId,Type,Date,FeeDetail,Fee amount (CCY),Fee currency,Link Id
0,FEE00000001,Deposit,2025-10-24,Flat Fee,4.89,GBP,371637
1,FEE00000002,Deposit,2025-10-24,% Fee,16.11,GBP,371637
2,FEE00000003,Deposit,2025-10-31,Flat Fee,17.79,GBP,374086
3,FEE00000004,Deposit,2025-10-31,% Fee,33.21,GBP,374086
4,FEE00000005,Deposit,2025-10-14,Flat Fee,20.00,USD,367878


### 1i. Do fees link to transactions? `Fees.Link Id` → `Transaction ID`
Fee revenue must attach to a transaction. Check how many `Link Id`s resolve to a known transaction, and the fee fan-out per transaction.

In [16]:
all_txn_ids = set(dep_ids) | set(wd_ids)
link = fees["Link Id"]
matched = link.isin(all_txn_ids)
print("Fee rows            :", len(fees))
print("Link Id matches a txn:", matched.sum(), f"({100*matched.mean():.1f}%)")
print("Unmatched Link Ids   :", (~matched).sum())
print("Distinct txns with a fee:", link[matched].nunique(), "of", len(all_txn_ids), "txns")
print("\nFees per transaction (distribution):")
fees[matched].groupby("Link Id").size().value_counts().sort_index().rename("num_transactions").to_frame().rename_axis("fees_per_txn")

Fee rows            : 21921
Link Id matches a txn: 21921 (100.0%)
Unmatched Link Ids   : 0
Distinct txns with a fee: 12676 of 12676 txns

Fees per transaction (distribution):


,num_transactions
fees_per_txn,
1,3856
2,8605
3,5
4,210


## 2. `companies.json` — direct companies (`dcId`)

Records live under `records`. Each is deeply nested (`registration`, `relationships`, `classification`,
`financials`, `footprint`, `attributes`). The bridge to the hierarchy is
`relationships.parentGroup.value → group.groupId`.

In [17]:
companies_meta = json.load(open(DATA / "companies.json"))["metadata"]
print(json.dumps(companies_meta, indent=1))

comp = con.sql(f"""
    SELECT
        r.dcId,
        r.registration.legalName            AS legal_name,
        r.registration.status               AS status,
        r.registration.incorporation.country AS incorp_country,
        r.relationships.parentGroup.value   AS parent_group_id,
        r.relationships.parentGroup.role    AS parent_role,
        r.classification.vertical           AS vertical,
        r.classification.industry           AS industry,
        r.regulatory.licensed               AS licensed,
        r.workforce.headcount               AS headcount,
        r.financials.currency               AS fin_currency,
        r.financials.annualRevenue          AS annual_revenue,
        r.footprint.operationCount          AS operation_count
    FROM (SELECT unnest(records) AS r FROM read_json_auto('{DATA / "companies.json"}'))
""").df()
profile_frame(comp, "companies (flattened)")

{
 "schema": "dc/v2",
 "generated_at": "2026-06-16T00:00:00Z",
 "entity_type": "direct_company",
 "natural_key": "dcId",
 "bridge_key": "relationships.parentGroup.value -> group.groupId",
 "total": 44
}
companies (flattened): 44 rows × 13 cols


,column,dtype,non_null,null_%,distinct,sample_values
0,dcId,str,44,0.0,44,"234850008297, 234939159769, 234854669542"
1,legal_name,str,44,0.0,35,"Payment One Holding S.à r.l., Payment One Operating Comp..."
2,status,str,44,0.0,3,"ACTIVE, SUSPENDED, UNKNOWN"
3,incorp_country,str,44,0.0,13,"Cyprus, Lithuania, Curaçao"
4,parent_group_id,str,44,0.0,13,"167898894567, 167986897142, 187752508616"
5,parent_role,str,44,0.0,6,"HOLDING, OPERATING, PAYMENTS"
6,vertical,str,44,0.0,8,"Holding, MSB, Gaming"
7,industry,str,44,0.0,8,"Holding Company / Group Treasury, Money Service Business..."
8,licensed,bool,44,0.0,2,"False, True"
9,headcount,int64,44,0.0,32,"25, 490, 30"


In [18]:
comp.head(10)

,dcId,legal_name,status,incorp_country,parent_group_id,parent_role,vertical,industry,licensed,headcount,fin_currency,annual_revenue,operation_count
0,234850008297,Payment One Holding S.à r.l.,ACTIVE,Cyprus,167898894567,HOLDING,Holding,Holding Company / Group Treasury,False,25,USD,USD 5.95M,6
1,234939159769,Payment One Operating Company Ltd,ACTIVE,Cyprus,167898894567,OPERATING,MSB,Money Service Business,True,490,USD,USD 91.70M,6
2,234854669542,Remitance Central Holdings Ltd,SUSPENDED,Lithuania,167986897142,HOLDING,Holding,Holding Company / Group Treasury,False,30,USD,USD 10.24M,4
3,234939159775,Remitance Central (Europe) Ltd,ACTIVE,Curaçao,167986897142,OPERATING,MSB,Money Service Business,True,600,USD,USD 303.60M,2
4,258599414982,Get Tranzzact (Europe) Ltd,ACTIVE,Luxembourg,187752508616,OPERATING,MSB,Money Service Business,True,395,USD,USD 171.08M,1
5,258991877340,ConstructGame Operating Company Ltd,ACTIVE,Netherlands,187754310858,OPERATING,Gaming,Online Gaming & Gambling,True,360,USD,USD 245.07M,2
6,258970163419,ConstructGame Payments UAB,ACTIVE,Gibraltar,187754310858,PAYMENTS,Payments,Payment & E-Money Services,True,200,USD,USD 129.64M,6
7,258740050135,ConstructGame Technology Pvt Ltd,ACTIVE,Curaçao,187754310858,TECHNOLOGY,Technology,Software Development & Platform Engineering,False,365,USD,USD 42.56M,6
8,258929414331,ConstructGame Media Ltd,SUSPENDED,Gibraltar,187754310858,MARKETING,Marketing,Marketing & Customer Acquisition,False,60,USD,USD 26.44M,4
9,279464884442,ConstructGame IP Ltd,SUSPENDED,Estonia,187754310858,LICENSING_IP,Licensing,Intellectual Property Licensing,False,5,USD,USD 42.87M,5


### 2a. companies — keys & hierarchy linkage

In [19]:
print("dcId distinct        :", comp["dcId"].nunique(), "of", len(comp), " dupes:", comp["dcId"].duplicated().sum())
print("parent_group_id nulls:", comp["parent_group_id"].isna().sum())
print("distinct parent groups:", comp["parent_group_id"].nunique())
print("statuses             :", comp["status"].value_counts().to_dict())
print("verticals            :", comp["vertical"].value_counts().to_dict())
print("annualRevenue format : free-text string, e.g.", comp["annual_revenue"].dropna().head(3).tolist(), "→ needs parsing if ever used")
print("companies per group  :")
comp.groupby("parent_group_id").size().sort_values(ascending=False).rename("num_companies").to_frame()

dcId distinct        : 44 of 44  dupes: 0
parent_group_id nulls: 0
distinct parent groups: 13
statuses             : {'ACTIVE': 35, 'SUSPENDED': 7, 'UNKNOWN': 2}
verticals            : {'Holding': 13, 'MSB': 8, 'Payments': 5, 'Technology': 5, 'Licensing': 4, 'Gaming': 3, 'Marketing': 3, 'Crypto': 3}
annualRevenue format : free-text string, e.g. ['USD 5.95M', 'USD 91.70M', 'USD 10.24M'] → needs parsing if ever used
companies per group  :


,num_companies
parent_group_id,
192230405343,18
187754310858,7
192400133342,5
167898894567,2
167986897142,2
192457113847,2
208399264973,2
187752508616,1
187757905092,1


## 3. `groups.json` — parent groups (`groupId`)

Groups live under `result.groups`. The `attributes` array is **heterogeneous**: `value` is sometimes a
scalar, sometimes an object (e.g. `country_of_incorporation → {label, iso2}`). We profile the scalar core
first, then inspect the attribute shapes separately — this is exactly the "handle scalar-or-object"
risk the spec calls out.

In [20]:
export_meta = json.load(open(DATA / "groups.json"))["export"]
print(json.dumps(export_meta, indent=1))

grp = con.sql(f"""
    SELECT
        g.groupId,
        g.profile.displayName                 AS display_name,
        g.profile.lifecycle.status.code        AS status_code,
        g.profile.lifecycle.status.active      AS active,
        g.segmentation.pod                     AS pod,
        g.segmentation.vertical                AS vertical,
        g.segmentation.industry                AS industry,
        g.segmentation.commercialTier          AS commercial_tier
    FROM (SELECT unnest(result.groups) AS g FROM read_json_auto('{DATA / "groups.json"}'))
""").df()
profile_frame(grp, "groups (flattened)")

{
 "schemaVersion": "1.3.0",
 "generatedAt": "2026-06-16T00:00:00Z",
 "entity": "group",
 "recordCount": 13,
 "naturalKey": "groupId"
}


groups (flattened): 13 rows × 8 cols


,column,dtype,non_null,null_%,distinct,sample_values
0,groupId,str,13,0.0,13,"167898894567, 167986897142, 187752508616"
1,display_name,str,13,0.0,13,"Payment One, Remitance Central, Get Tranzzact"
2,status_code,str,13,0.0,2,"ACTIVE, SUSPENDED"
3,active,bool,13,0.0,2,"True, False"
4,pod,str,13,0.0,3,"MSB/PSP, Gaming/iGaming/Gambling, Crypto/CFD/Forex"
5,vertical,str,13,0.0,3,"MSB, Gaming, Crypto"
6,industry,str,13,0.0,3,"Money Service Business, Online Gaming & Gambling, Crypto..."
7,commercial_tier,str,13,0.0,3,"Gold, Bronze, Silver"


In [21]:
grp

,groupId,display_name,status_code,active,pod,vertical,industry,commercial_tier
0,167898894567,Payment One,ACTIVE,True,MSB/PSP,MSB,Money Service Business,Gold
1,167986897142,Remitance Central,ACTIVE,True,MSB/PSP,MSB,Money Service Business,Bronze
2,187752508616,Get Tranzzact,SUSPENDED,False,MSB/PSP,MSB,Money Service Business,Gold
3,187754310858,ConstructGame,ACTIVE,True,Gaming/iGaming/Gambling,Gaming,Online Gaming & Gambling,Gold
4,187757905092,Sanno Payments,ACTIVE,True,MSB/PSP,MSB,Money Service Business,Silver
5,192212183238,Tech Key,ACTIVE,True,MSB/PSP,MSB,Money Service Business,Bronze
6,192230405343,Koba2Maya,ACTIVE,True,Gaming/iGaming/Gambling,Gaming,Online Gaming & Gambling,Silver
7,192315275482,Oni Group,ACTIVE,True,MSB/PSP,MSB,Money Service Business,Bronze
8,192400133342,NXT Services,ACTIVE,True,Crypto/CFD/Forex,Crypto,Crypto Assets & Leveraged Trading,Silver
9,192402023626,Cube Empire,SUSPENDED,False,MSB/PSP,MSB,Money Service Business,Bronze


### 3a. groups — heterogeneous `attributes` shapes
Unnest every attribute, keep the value as JSON, and tally which `valueType` / JSON shape each attribute name takes. Anything appearing as both scalar and object needs branching logic in Silver.

In [22]:
attrs = con.sql(f"""
    SELECT
        g.groupId,
        a.name       AS attr_name,
        a.valueType  AS value_type,
        json_type(to_json(a.value)) AS json_type,
        to_json(a.value) AS value_json
    FROM (SELECT unnest(result.groups) AS g FROM read_json_auto('{DATA / "groups.json"}')),
         unnest(g.attributes) AS t(a)
""").df()
print("Total attribute rows:", len(attrs))
shape = attrs.groupby(["attr_name", "value_type", "json_type"]).size().rename("n").reset_index()
display(shape)
mixed = shape.groupby("attr_name")["json_type"].nunique()
print("\nAttributes with MORE THAN ONE json shape (need scalar-or-object handling):",
      mixed[mixed > 1].index.tolist() or "none")

Total attribute rows: 26


,attr_name,value_type,json_type,n
0,country_of_incorporation,object,OBJECT,13
1,risk_level,string,VARCHAR,13



Attributes with MORE THAN ONE json shape (need scalar-or-object handling): none


## 4. `exchange_rates.json` — time-versioned FX to GBP

Structure: `{meta, rates.series.<CCY>.points}`. Each point is a **positional tuple**
`[rateId, validFromEpochMs, validTillEpochMs, rateStr, rateMantissaE10]`, **not pre-sorted**. The as-of
join finds the point where `validFrom <= t < validTill` and multiplies by `float(rateStr)`. This section
profiles coverage, per-currency point counts, sortedness, and — critically — **gaps/overlaps** in each
currency's timeline, since a transaction landing in a gap must be quarantined, not mis-priced.

In [23]:
fx = json.load(open(DATA / "exchange_rates.json"))
meta = fx["meta"]
print(json.dumps(meta, indent=1))
cov_from = pd.to_datetime(meta["coverage"]["fromEpochMs"], unit="ms", utc=True)
cov_till = pd.to_datetime(meta["coverage"]["tillEpochMs"], unit="ms", utc=True)
print("\nCoverage:", cov_from, "→", cov_till)
print("Currencies in file:", meta["currencyCount"], "| declared rowCount:", meta["rowCount"])
print("GBP present as a series?", "GBP" in fx["rates"]["series"], "(expected False → rate is 1.0)")

{
 "schema": "fx-history/v2",
 "generatedAt": "2026-06-16T00:00:00Z",
 "base": "GBP",
 "direction": "source_to_base",
 "rowCount": 161342,
 "currencyCount": 36,
 "coverage": {
  "fromEpochMs": 1750061645260,
  "tillEpochMs": 1768204893003
 },
 "tuple": {
  "fieldOrder": [
   "rateId",
   "validFromEpochMs",
   "validTillEpochMs",
   "rateStr",
   "rateMantissaE10"
  ],
  "rateScale": 10,
  "timestampUnit": "epoch_ms_utc"
 },
 "usage": "To convert X units of <CCY> to GBP at instant t (epoch ms): take rates['series'][<CCY>], find the point where validFromEpochMs <= t < validTillEpochMs, then GBP = X * float(rateStr). Points are NOT pre-sorted. Rates only cover the window in meta.coverage; transactions outside it have no direct rate."
}

Coverage: 2025-06-16 08:14:05.260000+00:00 → 2026-01-12 08:01:33.003000+00:00
Currencies in file: 36 | declared rowCount: 161342
GBP present as a series? False (expected False → rate is 1.0)


### 4a. Per-currency point counts, rate range, and timeline integrity
For each currency we flatten its points, check the validity intervals are contiguous (no gaps, no overlaps), and record the min/max rate. Gaps/overlaps are where the as-of join can silently go wrong.

In [24]:
fo = meta["tuple"]["fieldOrder"]  # ['rateId','validFromEpochMs','validTillEpochMs','rateStr','rateMantissaE10']
recs = []
for ccy, series in fx["rates"]["series"].items():
    pts = series["points"]
    df = pd.DataFrame(pts, columns=fo)
    df["validFromEpochMs"] = df["validFromEpochMs"].astype("int64")
    df["validTillEpochMs"] = df["validTillEpochMs"].astype("int64")
    df["rate"] = df["rateStr"].astype(str).str.strip('"').astype(float)
    df = df.sort_values("validFromEpochMs").reset_index(drop=True)
    # gaps / overlaps: does each interval's end meet the next interval's start?
    gaps = (df["validFromEpochMs"].shift(-1) > df["validTillEpochMs"]).sum()
    overlaps = (df["validFromEpochMs"].shift(-1) < df["validTillEpochMs"]).sum()
    pre_sorted = list(pd.DataFrame(pts, columns=fo)["validFromEpochMs"].astype("int64")) == \
                 sorted(df["validFromEpochMs"].tolist())
    recs.append({
        "ccy": ccy, "points": len(df),
        "from": pd.to_datetime(df["validFromEpochMs"].min(), unit="ms", utc=True).date(),
        "till": pd.to_datetime(df["validTillEpochMs"].max(), unit="ms", utc=True).date(),
        "rate_min": round(df["rate"].min(), 6), "rate_max": round(df["rate"].max(), 6),
        "gaps": int(gaps), "overlaps": int(overlaps), "pre_sorted": pre_sorted,
    })
fx_summary = pd.DataFrame(recs).sort_values("points", ascending=False).reset_index(drop=True)
print("Total points across all currencies:", fx_summary["points"].sum(), "(meta rowCount:", meta["rowCount"], ")")
print("Any currency pre-sorted?          :", fx_summary["pre_sorted"].any(), "→ must sort/range-match ourselves")
print("Currencies with gaps              :", fx_summary.loc[fx_summary["gaps"]>0, "ccy"].tolist() or "none")
print("Currencies with overlaps          :", fx_summary.loc[fx_summary["overlaps"]>0, "ccy"].tolist() or "none")
fx_summary

Total points across all currencies: 161342 (meta rowCount: 161342 )
Any currency pre-sorted?          : False → must sort/range-match ourselves
Currencies with gaps              : none
Currencies with overlaps          : none


,ccy,points,from,till,rate_min,rate_max,gaps,overlaps,pre_sorted
0,EUR,50165,2025-06-30,2026-01-01,0.855802,0.886453,0,0,False
1,USD,44480,2025-06-30,2026-01-01,0.725271,0.768460,0,0,False
2,CAD,11850,2025-06-30,2026-01-01,0.528661,0.548276,0,0,False
3,NOK,8651,2025-06-30,2026-01-01,0.072028,0.076068,0,0,False
4,CHF,8074,2025-06-30,2026-01-01,0.912545,0.963883,0,0,False
5,NZD,7551,2025-06-30,2026-01-01,0.425037,0.448298,0,0,False
6,HKD,6359,2025-06-30,2026-01-01,0.092442,0.098828,0,0,False
7,ZAR,6329,2025-06-30,2026-01-01,0.041089,0.044950,0,0,False
8,CNY,6239,2025-06-30,2026-01-01,0.101328,0.108026,0,0,False
9,SEK,3976,2025-06-30,2026-01-01,0.076383,0.080966,0,0,False


## 5. Cross-source integrity — the questions that decide the model

The pipeline stands or falls on these joins. We check each linkage the Gold network needs:

1. **FX feasibility** — is every transaction/fee currency actually priced, and does the transaction
   window sit inside FX coverage?
2. **Counterparty → group / company** — can we resolve counterparts to a group node (circle) vs a
   standalone node (diamond)?
3. **Company → group** — does every transacting company roll up to a known group?

### 5a. FX feasibility — currencies used vs currencies priced, window vs coverage

In [25]:
priced = set(fx["rates"]["series"].keys()) | {"GBP"}  # GBP implicit at 1.0
tx_ccy = set(sheets["Deposit"]["Tx Currency"].dropna()) | set(sheets["Withdrawals"]["Tx Currency"].dropna())
fee_ccy = set(sheets["Fees"]["Fee currency"].dropna())
print("Transaction currencies:", sorted(tx_ccy))
print("Fee currencies        :", sorted(fee_ccy))
print("Unpriced tx currencies :", sorted(tx_ccy - priced) or "none — all priced ✅")
print("Unpriced fee currencies:", sorted(fee_ccy - priced) or "none — all priced ✅")

# window vs coverage. Tx Date is a midnight datetime, Tx Time a time string; combine as
# date + timedelta (robust to rows with/without fractional seconds — a plain string concat +
# to_datetime locks onto the first row's format and coerces the rest to NaT).
def tx_ts(df):
    base = pd.to_datetime(df["Tx Date"], errors="coerce").dt.normalize()
    td = pd.to_timedelta(df["Tx Time"].astype(str), errors="coerce")
    return (base + td).dt.tz_localize("UTC")
dep_ts, wd_ts = tx_ts(sheets["Deposit"]), tx_ts(sheets["Withdrawals"])
all_ts = pd.concat([dep_ts, wd_ts])
print("\nTxn timestamp range :", all_ts.min(), "→", all_ts.max())
print("FX coverage range   :", cov_from, "→", cov_till)
out_of_cov = ((all_ts < cov_from) | (all_ts >= cov_till)).sum()
print("Txns outside FX coverage:", out_of_cov, "→ would be quarantined")
print("Unparseable timestamps  :", all_ts.isna().sum())

Transaction currencies: ['AED', 'AUD', 'CAD', 'CHF', 'DKK', 'EUR', 'GBP', 'NOK', 'SEK', 'USD']
Fee currencies        : ['AED', 'CAD', 'CHF', 'DKK', 'EUR', 'GBP', 'NOK', 'SEK', 'USD']
Unpriced tx currencies : none — all priced ✅
Unpriced fee currencies: none — all priced ✅

Txn timestamp range : 2025-07-01 04:30:31.990000+00:00 → 2025-12-31 17:38:56.953000+00:00
FX coverage range   : 2025-06-16 08:14:05.260000+00:00 → 2026-01-12 08:01:33.003000+00:00
Txns outside FX coverage: 0 → would be quarantined
Unparseable timestamps  : 0


### 5b. Counterparty → group / company linkage
The network needs each counterpart resolved to a group (circle) or left standalone (diamond). How many `CP ID`s used in transactions resolve, and do the `Group ID` / `DC Id` on the Counterparty sheet actually match `groups.json` / `companies.json`?

In [26]:
cp = sheets["Counterparty"].copy()
cp["Group ID"] = cp["Group ID"].astype("Int64").astype(str).replace("<NA>", None)
cp["DC Id"]    = cp["DC Id"].astype("Int64").astype(str).replace("<NA>", None)
group_ids = set(grp["groupId"].astype(str))
dc_ids    = set(comp["dcId"].astype(str))

used_cp = set(sheets["Deposit"]["Counterparty"].dropna()) | set(sheets["Withdrawals"]["Counterparty"].dropna())
known_cp = set(cp["CP ID"])
print("Distinct CP IDs used in txns :", len(used_cp))
print("  resolvable in Counterparty :", len(used_cp & known_cp))
print("  NOT in Counterparty sheet  :", len(used_cp - known_cp), (sorted(used_cp - known_cp)[:10] if used_cp - known_cp else ""))

print("\nCounterparty rows total      :", len(cp))
print("  with a Group ID            :", cp['Group ID'].notna().sum())
print("    Group ID ∈ groups.json   :", cp['Group ID'].dropna().isin(group_ids).sum(),
      "  orphan:", (~cp['Group ID'].dropna().isin(group_ids)).sum())
print("  with a DC Id               :", cp['DC Id'].notna().sum())
print("    DC Id ∈ companies.json   :", cp['DC Id'].dropna().isin(dc_ids).sum(),
      "  orphan:", (~cp['DC Id'].dropna().isin(dc_ids)).sum())
print("  standalone (no group/dc)   :", ((cp['Group ID'].isna()) & (cp['DC Id'].isna())).sum(), "→ diamond nodes")

Distinct CP IDs used in txns : 1585
  resolvable in Counterparty : 1585
  NOT in Counterparty sheet  : 0 

Counterparty rows total      : 1585
  with a Group ID            : 222
    Group ID ∈ groups.json   : 222   orphan: 0
  with a DC Id               : 222
    DC Id ∈ companies.json   : 222   orphan: 0
  standalone (no group/dc)   : 1363 → diamond nodes


### 5c. Transacting company → group rollup
Fact rows carry `Deal ID/DC ID`. Does each resolve to a company in `companies.json`, and thence to a group? This is the drill-up path (company → group).

In [27]:
dep_dc = sheets["Deposit"]["Deal ID/DC ID"].astype("Int64").astype(str)
wd_dc  = sheets["Withdrawals"]["Deal ID/DC ID"].astype("Int64").astype(str)
used_dc = set(dep_dc) | set(wd_dc)
print("Distinct Deal ID/DC ID in facts:", len(used_dc))
print("  ∈ companies.json (dcId)      :", len(used_dc & dc_ids))
print("  NOT in companies.json        :", len(used_dc - dc_ids), (sorted(used_dc - dc_ids)[:10] if used_dc - dc_ids else ""))
comp_to_grp = dict(zip(comp["dcId"].astype(str), comp["parent_group_id"].astype(str)))
resolvable_grp = {comp_to_grp.get(d) for d in used_dc if d in comp_to_grp}
print("  → distinct parent groups they roll up to:", len({g for g in resolvable_grp if g and g != 'None'}))

Distinct Deal ID/DC ID in facts: 34
  ∈ companies.json (dcId)      : 34
  NOT in companies.json        : 0 
  → distinct parent groups they roll up to: 13


## 6. Profiling summary — findings & likely data-quality actions

Fill/confirm against the cell outputs above; the numbers are printed so this reads as evidence, not
assertion. Framed as **dropped / quarantined / kept** per the brief's DQ requirement.

In [28]:
findings = [
    ("Deposit / Withdrawals column order differs (same set)", "KEEP — union by column name, never position (per build protocol)"),
    ("Transaction ID uniqueness & cross-sheet overlap",        "see §1f — a shared ID would need disambiguation"),
    ("Fees.Link Id → Transaction ID match rate",               "see §1i — unmatched fees can't attach revenue → quarantine"),
    ("Counterparty rows with no Group ID / DC Id",             "KEEP as standalone 'diamond' nodes (expected, not an error)"),
    ("CP IDs used in txns but absent from Counterparty sheet", "see §5b — unresolved counterparts, quarantine or label unknown"),
    ("All transaction/fee currencies priced in FX file",       "see §5a — any unpriced currency → quarantine"),
    ("Transactions outside FX coverage window",                "see §5a — quarantine with reason, never mis-price"),
    ("FX points not pre-sorted; per-ccy gaps/overlaps",        "see §4a — sort + range-match; gap instants → quarantine"),
    ("Heterogeneous group attributes (scalar vs object)",      "see §3a — branch on json_type in Silver"),
    ("annualRevenue is free-text ('USD 5.95M')",               "out of scope for network; parse only if ever needed"),
    ("Non-positive / null Tx values or fee amounts",           "see §1e/§1h — review before summing into edges"),
]
pd.DataFrame(findings, columns=["observation", "suggested_action"])

,observation,suggested_action
0,Deposit / Withdrawals column order differs (same set),"KEEP — union by column name, never position (per build p..."
1,Transaction ID uniqueness & cross-sheet overlap,see §1f — a shared ID would need disambiguation
2,Fees.Link Id → Transaction ID match rate,see §1i — unmatched fees can't attach revenue → quarantine
3,Counterparty rows with no Group ID / DC Id,"KEEP as standalone 'diamond' nodes (expected, not an error)"
4,CP IDs used in txns but absent from Counterparty sheet,"see §5b — unresolved counterparts, quarantine or label u..."
5,All transaction/fee currencies priced in FX file,see §5a — any unpriced currency → quarantine
6,Transactions outside FX coverage window,"see §5a — quarantine with reason, never mis-price"
7,FX points not pre-sorted; per-ccy gaps/overlaps,see §4a — sort + range-match; gap instants → quarantine
8,Heterogeneous group attributes (scalar vs object),see §3a — branch on json_type in Silver
9,annualRevenue is free-text ('USD 5.95M'),out of scope for network; parse only if ever needed


In [29]:
con.close()
print("Profiling complete. No warehouse tables were written — this notebook is read-only.")

Profiling complete. No warehouse tables were written — this notebook is read-only.
